# Judge analysis of the validation split

The same judge rates as `judge.ipynb`, for the held-out validation split.

In [1]:
import os
from pathlib import Path

# Paths below are relative to the repo root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "pyproject.toml").is_file(), f"unexpected cwd: {REPO_ROOT}"

import pandas as pd

from grant_eval.config import MODEL_ROUTING
from grant_eval.scoring import judge as judgemod
from grant_eval.scoring import judge_notebook as jn

#: The held-out pass, written by
#:     uv run judge eval/validation/grid/ --results-root eval/judge_results_validation
RESULTS_ROOT = Path("eval/judge_results_validation")

## Provenance

In [2]:
judge_config = judgemod.JudgeConfig(
    model=judgemod.JUDGE_MODEL, temperature=0.0,
    prompt_version=judgemod.PROMPT_VERSION, rubric_version=judgemod.RUBRIC_VERSION,
)
rollup_out_path = judgemod.rollup_path(judge_config.config_hash, root=RESULTS_ROOT)
assert rollup_out_path.is_file(), (
    f"no roll-up at {rollup_out_path}.\n"
    "Either the validation split has not been judged under this judge config, or the "
    "judge config has changed since it was (model / temperature / prompt_version / "
    "rubric_version). Re-run:\n"
    "    uv run judge eval/validation/grid/ --results-root eval/judge_results_validation"
)
rollup = judgemod.read_verdict_file(rollup_out_path)

print("judge model       :", judge_config.model)
print("provider pin      :", MODEL_ROUTING["gemini-judge"]["provider"])
print("temperature       :", judge_config.temperature)
print("prompt version    :", judge_config.prompt_version)
print("rubric version    :", judge_config.rubric_version)
print("-> config hash    :", judge_config.config_hash)
print()
print("roll-up           :", rollup_out_path.name)
print("runs in roll-up   :", rollup["n_runs"])
print("judged fields     :", ", ".join(judgemod.JUDGED_FIELDS))
print("NOT judged        : discipline_tags (a tag is a classification, not a claim)")
print("dimensions        :", ", ".join(judgemod.DIMENSIONS))

judge model       : openrouter/google/gemini-2.5-flash-lite
provider pin      : {'only': ['google-ai-studio'], 'allow_fallbacks': False}
temperature       : 0.0
prompt version    : v1
rubric version    : v1
-> config hash    : 17fb8f1bc516e87749f31b1c17dd772bd675dc99b9e6ea3c77ecb4f0636663a1

roll-up           : rollup-17fb8f1bc516e87749f31b1c17dd772bd675dc99b9e6ea3c77ecb4f0636663a1.json
runs in roll-up   : 80
judged fields     : summary, eligibility_summary
NOT judged        : discipline_tags (a tag is a classification, not a claim)
dimensions        : completeness, groundedness


In [3]:
# Same glob as judge.ipynb: verdicts are stored at
# <run_config_hash>/<funder_slug>/<replicate>/<judge_config_hash>.json, so the judge
# config hash is the leaf name and one config's corpus is one glob.
verdict_paths = sorted(RESULTS_ROOT.glob(f"*/*/*/{judge_config.config_hash}.json"))
arms = sorted({p.parts[-4] for p in verdict_paths})
funders = sorted({p.parts[-3] for p in verdict_paths})
replicates = sorted({p.parts[-2] for p in verdict_paths})

print(f"verdict files     : {len(verdict_paths)}")
print(f"locked arms       : {len(arms)}  (model x prompt strategy x temperature)")
print(f"funders           : {len(funders)}")
print(f"replicates        : {', '.join(replicates)}")
print()
for funder in funders:
    print(" ", funder)

verdict files     : 80
locked arms       : 2  (model x prompt strategy x temperature)
funders           : 8
replicates        : 1, 2, 3, 4, 5

  fulbright
  hakluyt
  heritage-fund
  pathological-society
  pilgrim-trust
  prs-foundation
  royal-hist-soc
  society-of-authors


## Claim-level and record-level rates by field and dimension

In [4]:
def rate_or_undefined(cell, key):
    return "undefined" if cell[f"{key}_undefined"] else round(cell[key], 3)


rates_df = pd.DataFrame([
    {
        "field:dimension": key,
        "n_claims": cell["n_claims"], "n_records_judged": cell["n_records_judged"],
        "claim_rate (micro)": rate_or_undefined(cell, "claim_rate"),
        "claim_rate (macro)": rate_or_undefined(cell, "claim_rate_macro"),
        "record_rate": rate_or_undefined(cell, "record_rate"),
        "n_raw_html_fallback": cell["n_raw_html_fallback"],
    }
    for key, cell in sorted(rollup["fields"].items())
]).set_index("field:dimension")
rates_df

,n_claims,n_records_judged,claim_rate (micro),claim_rate (macro),record_rate,n_raw_html_fallback
field:dimension,,,,,,
eligibility_summary:completeness,2762,523,0.682,0.738,0.409,0
eligibility_summary:groundedness,2196,566,0.966,0.947,0.929,0
summary:completeness,3223,522,0.742,0.733,0.209,0
summary:groundedness,4381,571,0.993,0.993,0.954,0


## Rates by run attribute

In [5]:
run_dicts = [judgemod.read_verdict_file(p) for p in verdict_paths]
print(f"{len(run_dicts)} run verdict file(s) loaded from disk")

#: (roll-up cell key, column label) for each of the three rates.
RATE_KEYS = [
    ("claim_rate", "micro claim rate"),
    ("claim_rate_macro", "macro claim rate"),
    ("record_rate", "record rate"),
]


def breakdown(attribute):
    grouped = jn.group_rollup(run_dicts, key=attribute)
    rows = []
    for group, cells in sorted(grouped.items()):
        for field_dim, cell in sorted(cells.items()):
            row = {attribute: group, "field:dimension": field_dim,
                   "n_claims": cell["n_claims"]}
            for key, label in RATE_KEYS:
                row[label] = rate_or_undefined(cell, key)
            rows.append(row)
    return pd.DataFrame(rows).set_index([attribute, "field:dimension"])

80 run verdict file(s) loaded from disk


In [6]:
breakdown("prompt_strategy")

n_claims  micro claim rate  \
prompt_strategy field:dimension                                                
few_shot        eligibility_summary:completeness      1362             0.680   
                eligibility_summary:groundedness      1128             0.960   
                summary:completeness                  1622             0.726   
                summary:groundedness                  2282             0.996   
minimal         eligibility_summary:completeness      1400             0.684   
                eligibility_summary:groundedness      1068             0.972   
                summary:completeness                  1601             0.759   
                summary:groundedness                  2099             0.990   

                                                  macro claim rate  \
prompt_strategy field:dimension                                      
few_shot        eligibility_summary:completeness             0.736   
                eligibility_summary:groundedness             0.931   
                summary:completeness                         0.721   
                summary:groundedness                         0.995   
minimal         eligibility_summary:completeness             0.740   
                eligibility_summary:groundedness             0.963   
                summary:completeness                         0.745   
                summary:groundedness                         0.991   

                                                  record rate  
prompt_strategy field:dimension                                
few_shot        eligibility_summary:completeness        0.418  
                eligibility_summary:groundedness        0.914  
                summary:completeness                    0.194  
                summary:groundedness                    0.966  
minimal         eligibility_summary:completeness        0.400  
                eligibility_summary:groundedness        0.946  
                summary:completeness                    0.224  
                summary:groundedness                    0.942

In [7]:
breakdown("model")

n_claims  \
model                                      field:dimension                              
openrouter/deepseek/deepseek-v4-flash-0731 eligibility_summary:completeness      2762   
                                           eligibility_summary:groundedness      2196   
                                           summary:completeness                  3223   
                                           summary:groundedness                  4381   

                                                                             micro claim rate  \
model                                      field:dimension                                      
openrouter/deepseek/deepseek-v4-flash-0731 eligibility_summary:completeness             0.682   
                                           eligibility_summary:groundedness             0.966   
                                           summary:completeness                         0.742   
                                           summary:groundedness                         0.993   

                                                                             macro claim rate  \
model                                      field:dimension                                      
openrouter/deepseek/deepseek-v4-flash-0731 eligibility_summary:completeness             0.738   
                                           eligibility_summary:groundedness             0.947   
                                           summary:completeness                         0.733   
                                           summary:groundedness                         0.993   

                                                                             record rate  
model                                      field:dimension                                
openrouter/deepseek/deepseek-v4-flash-0731 eligibility_summary:completeness        0.409  
                                           eligibility_summary:groundedness        0.929  
                                           summary:completeness                    0.209  
                                           summary:groundedness                    0.954

In [8]:
breakdown("temperature")

n_claims  micro claim rate  \
temperature field:dimension                                                
0.7         eligibility_summary:completeness      2762             0.682   
            eligibility_summary:groundedness      2196             0.966   
            summary:completeness                  3223             0.742   
            summary:groundedness                  4381             0.993   

                                              macro claim rate  record rate  
temperature field:dimension                                                  
0.7         eligibility_summary:completeness             0.738        0.409  
            eligibility_summary:groundedness             0.947        0.929  
            summary:completeness                         0.733        0.209  
            summary:groundedness                         0.993        0.954

## By funder

In [9]:
by_funder_rows = []
for funder, cells in sorted(rollup["by_funder"].items()):
    for field_dim, cell in sorted(cells.items()):
        row = {"funder": funder, "field:dimension": field_dim,
               "n_claims": cell["n_claims"]}
        for key, label in RATE_KEYS:
            row[label] = rate_or_undefined(cell, key)
        by_funder_rows.append(row)

pd.DataFrame(by_funder_rows).set_index(["funder", "field:dimension"])

n_claims  \
funder                                             field:dimension                              
Hakluyt Society                                    eligibility_summary:completeness        44   
                                                   eligibility_summary:groundedness        41   
                                                   summary:completeness                    65   
                                                   summary:groundedness                    77   
National Lottery Heritage Fund                     eligibility_summary:completeness        75   
                                                   eligibility_summary:groundedness       130   
                                                   summary:completeness                    65   
                                                   summary:groundedness                   141   
PRS Foundation                                     eligibility_summary:completeness       397   
                                                   eligibility_summary:groundedness       246   
                                                   summary:completeness                   582   
                                                   summary:groundedness                   582   
Royal Historical Society                           eligibility_summary:completeness       326   
                                                   eligibility_summary:groundedness       259   
                                                   summary:completeness                   285   
                                                   summary:groundedness                   733   
Society of Authors                                 eligibility_summary:completeness       471   
                                                   eligibility_summary:groundedness       338   
                                                   summary:completeness                   447   
                                                   summary:groundedness                   713   
The Pathological Society of Great Britain and I... eligibility_summary:completeness       953   
                                                   eligibility_summary:groundedness       827   
                                                   summary:completeness                  1221   
                                                   summary:groundedness                  1488   
The Pilgrim Trust                                  eligibility_summary:completeness       327   
                                                   eligibility_summary:groundedness       257   
                                                   summary:completeness                   350   
                                                   summary:groundedness                   407   
US-UK Fulbright Commission                         eligibility_summary:completeness       169   
                                                   eligibility_summary:groundedness        98   
                                                   summary:completeness                   208   
                                                   summary:groundedness                   240   

                                                                                     micro claim rate  \
funder                                             field:dimension                                      
Hakluyt Society                                    eligibility_summary:completeness             0.614   
                                                   eligibility_summary:groundedness             0.951   
                                                   summary:completeness                         0.877   
                                                   summary:groundedness                         0.935   
National Lottery Heritage Fund                     eligibility_summary:completeness             0.600   
                                                   